# 3D Spatial Reasoning using 5 views

Task: Given 5 orthographic views (front, back, top, bottom and left) of a 3D voxel scene, conclude what the right view must look like.

In [ ]:
import kaggle_benchmarks as kbench
import json
from typing import Dict
import pandas as pd
import re
import time

## Input constants
Input dataset has 200 tasks maximum. TASKS_SIZE selects range of tasks. Use 3 to 5 tasks for debugging, 50 or 100 tasks for benchmark evaluation.

In [ ]:
TASKS_SIZE = 50
TASK_NAME = f"5 views input, fixed size 3"
DATASET_VARIANT = "5v_fixed_3"

In [ ]:
# See https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
# to select a different model to test
# Use `kbench.llms` to list all available models
# `kbench.llm` is the default Kaggle defined model

USE_MODEL = kbench.llm #kbench.llms["anthropic/claude-opus-4-6@default"]

## Supporting functions

In [ ]:
def format_input_views(input_views: Dict[str, list[list[int]]]) -> str:
    """
    Format all input views as text.
    
    Args:
        input_views: Dictionary containing all 5 input views
            - front: 2D grid for front view
            - back: 2D grid for back view
            - top: 2D grid for top view
            - bottom: 2D grid for bottom view
            - left: 2D grid for left view
        
    Returns:
        Formatted text showing all views
    """
    # Define view names and their descriptions
    view_specs = [
        ("front", "FRONT VIEW (looking along +Y axis: X horizontal, Z vertical):"),
        ("back", "BACK VIEW (looking along -Y axis: X horizontal, Z vertical):"),
        ("top", "TOP VIEW (looking down -Z axis: X horizontal, Y vertical):"),
        ("bottom", "BOTTOM VIEW (looking up +Z axis: X horizontal, Y vertical):"),
        ("left", "LEFT VIEW (looking along +X axis: Y horizontal, Z vertical):")
    ]
    
    lines = []
    
    for view_key, view_label in view_specs:
        if view_key not in input_views:
            continue
        
        view_grid = input_views[view_key]
        
        # Format view as JSON-style array
        lines.append(view_label)
        lines.append("[")
        for i, row in enumerate(view_grid):
            row_str = " [" + ", ".join(str(cell) for cell in row) + "]"
            if i < len(view_grid) - 1:
                row_str += ","
            lines.append(row_str)
        lines.append("]")
        lines.append("")
    
    return "\n".join(lines)

In [ ]:
def build_prompt(
    input_views: Dict[str, list[list[int]]]) -> str:
    """
    Build the complete prompt text from task data.
    
    This helper function can be used to preview/export prompts without running the task.
    
    Args:
        input_views: Dictionary containing all input views
        

    Returns:
        Complete formatted prompt string
    """
    # Format the input views as text
    input_text = format_input_views(input_views)
    
    # Build and return the full prompt
    return f"""
You are given 5 orthographic views FRONT, BACK, TOP, BOTTOM, and LEFT of a 3D voxel scene. 
Each view shows the FIRST non-zero voxel encountered along the viewing direction.
Each cell contains a number (0-9). Number 0 represents an empty or transparent voxel.

Based on these 5 VIEWS, determine what the scene looks like from the RIGHT VIEW.

CRITICAL OUTPUT REQUIREMENTS:
- Output ONLY the JSON object. Nothing else.
- Do NOT include any explanation, reasoning, or commentary.
- Do NOT use LaTeX, markdown, or formatted text.
- Do NOT use phrases like "The final answer is" or similar.
- Your ENTIRE response must be ONLY: [[ number, number, ...], [...], ...]
- If you include ANY text besides the JSON, your answer will be marked INCORRECT.

## Camera positions

- FRONT VIEW
    - Camera position: Y = −∞
    - Looking direction: toward +Y
    - Projection rule: first non‑zero voxel along +Y
    - View orientation:
        - Row 0 = highest Z (top of the scene)
        - Column 0 = lowest X (left side of the scene)

- BACK VIEW
    - Camera position: Y = +∞
    - Looking direction: toward −Y
    - Projection rule: first non‑zero voxel along −Y
    - View orientation:
        - Row 0 = highest Z (top of the scene)
        - Column 0 = highest X (left side of the scene)

- TOP VIEW
    - Camera position: Z = +∞
    - Looking direction: toward -Z
    - Projection rule: first non‑zero voxel along -Z
    - View orientation:
        - Row 0 = highest Y (closest to back of the scene)
        - Column 0 = lowest X (left side of the scene)
        
- BOTTOM VIEW
    - Camera position: Z = −∞
    - Looking direction: toward +Z
    - Projection rule: first non‑zero voxel along +Z
    - View orientation:
        - Row 0 = lowest Y (closest to front of the scene)
        - Column 0 = lowest X (right side of the scene)

- LEFT VIEW
    - Camera position: X = −∞
    - Looking direction: toward +X
    - Projection rule: first non‑zero voxel along +X
    - View orientation:
        - Row 0 = highest Z (top of the scene)
        - Column 0 = highest Y (left side of the scene)

- RIGHT VIEW
    - Camera position: X = +∞
    - Looking direction: toward −X
    - Projection rule: first non‑zero voxel along −X
    - View orientation:
        - Row 0 = highest Z (top of the scene)
        - Column 0 = lowest Y (left side of the scene)

{input_text}

---

# EXAMPLE

Given:
FRONT VIEW:
[
  [5 0 6],
  [0 0 0],
  [1 4 2]
]

BACK VIEW:
[
  [8 0 7],
  [0 0 0],
  [2 4 3]
]

TOP VIEW:
[ 
  [7 4 8],
  [0 0 0],
  [5 0 6]
]

BOTTOM VIEW:
[
  [1 0 2],
  [0 0 0],
  [3 4 8]
]

LEFT VIEW:
[
  [7 0 5],
  [0 0 0],
  [3 0 1]
]

ANSWER for RIGHT VIEW:
  [
    [6 0 8], 
    [0 0 0], 
    [2 0 4]
  ]

Output: {{"answer": [[6, 0, 8], [0, 0, 0], [2, 0, 4]]}}

--- 

INSTRUCTIONS:
1. Analyze the FRONT view to understand which voxels exist at each (X, Z) position
2. Analyze the TOP view to understand which voxels exist at each (X, Y) position
3. Mentally reconstruct the full 3D scene by combining all views
4. Project the 3D scene onto the Y-Z plane (the right-side view)
5. Check the projected view against the camera position to determine the correct right-side view

RESPOND WITH ONLY THIS FORMAT (replace <list> with a 2D list representing the right view grid):
{{"answer": <list>}}

DO NOT INCLUDE:
- Explanations or reasoning
- LaTeX or mathematical notation  
- Natural language text
- Phrases like "The final answer is"
- Code blocks or markdown formatting

VALID RESPONSE EXAMPLES:
{{"answer": [[6, 0, 8], [0, 0, 0], [2, 0, 4]]}}
{{"answer": [[0, 0, 1, 1], [1, 3, 0, 1], [1, 2, 3, 2], [0, 0, 1, 0]]}}

Now output your answer:
"""

In [ ]:
def extract_answer_from_response(response: str) -> list[list[int]]:
    """Extract the answer grid from an LLM response."""

    # Try direct JSON extraction first (avoid extra judge LLM call)
    try:
        # Look for {"answer": [[...]]} pattern
        match = re.search(r'\{\s*"answer"\s*:\s*(\[\s*\[.*?\]\s*\])\s*\}', response, flags=re.DOTALL)
        if match:
            grid = json.loads(match.group(1))
            if isinstance(grid, list) and all(isinstance(row, list) for row in grid):
                return grid
        
        # Look for bare 2D array [[...]]
        match = re.search(r'(\[\s*\[.*?\]\s*\])', response, flags=re.DOTALL)
        if match:
            grid = json.loads(match.group(1))
            if isinstance(grid, list) and all(isinstance(row, list) for row in grid):
                return grid
    except (json.JSONDecodeError, AttributeError):
        pass
    
    # Fallback to judge LLM only if direct extraction failed
    prompt = f"""
Extract ONLY the answer grid from the response below.

Return STRICT JSON in this exact format:
{{"answer": <2D list of integers>}}

Do NOT include explanations, comments, or code fences.

Response to extract from:
{response}
"""

    try:
        judge_output = kbench.judge_llm.prompt(prompt)

        # Log raw judge output for debugging
        print("Judge output:", judge_output)

        # Extract the first {...} JSON object
        match = re.search(r"\{.*\}", judge_output, flags=re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in judge output")

        json_str = match.group(0)

        # Parse JSON
        parsed = json.loads(json_str)
        predicted_answer = parsed.get("answer", [])
        return predicted_answer

    except Exception as e:
        print(f"Judge LLM extraction failed: {e}")
        return []

In [ ]:
def strip_response(response: str) -> str:
    # Strip reasoning/thinking tags from various LLM makers
    # DeepSeek R1: <think>...</think>
    # Claude/others: <thinking>...</thinking>
    # Generic: <reasoning>...</reasoning>
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL | re.IGNORECASE)
    response = re.sub(r'<thinking>.*?</thinking>', '', response, flags=re.DOTALL | re.IGNORECASE)
    response = re.sub(r'<reasoning>.*?</reasoning>', '', response, flags=re.DOTALL | re.IGNORECASE)
    return response

In [ ]:
def check_answer(predicted: list[list[int]], correct: list[list[int]]) -> float:
    """Check if the predicted answer matches the correct answer or its mirrored version."""
    if predicted == correct:
        return 1.0
    
    # Check for horizontal mirroring
    mirrored = [row[::-1] for row in correct]
    if predicted == mirrored:
        return 0.33
    
    # Check for vertical mirroring
    mirrored = correct[::-1]
    if predicted == mirrored:
        return 0.33
    
    # Check for both horizontal and vertical mirroring
    mirrored = [row[::-1] for row in correct[::-1]]
    if predicted == mirrored:
        return 0.17
    
    return 0.0

## Benchmark task definition

In [ ]:
@kbench.task(name="single_task", store_task=False)
def single_task(llm, task) -> dict:
    # Initialize base result with common fields
    result = {
        "id": task['id'],
        "model": llm.name
    }
    
    try:
        print(f"##### Evaluating task id {task['id']} with {llm.name}")
        
        # Default LLM times out causing very long wait time after update
        if llm.name == "google/gemini-2.5-flash":
            result.update({
                "predicted_answer": "SKIPPED DEFAULT LLM",
                "time_elapsed": 0.0,
                "correctness": 0.0
            })
            return result

        prompt = build_prompt(input_views=task["input"])
        
        # Measure LLM response time
        llm_start = time.time()
        response = llm.prompt(prompt)
        llm_elapsed = time.time() - llm_start
        print(f"LLM response time: {llm_elapsed:.1f}s")
        response = strip_response(response)
        #print(f"LLM response: {response}")  # Uncomment only during debugging
        
        # Measure answer extraction time
        extraction_start = time.time()
        predicted_answer = extract_answer_from_response(response)
        extraction_elapsed = time.time() - extraction_start
        print(f"Answer extraction time: {extraction_elapsed:.1f}s")
        #print(f"Predicted answer: {predicted_answer}")  # Uncomment only during debugging

        # Check if prediction matches the correct answer (compare 2D grids)
        correctness = check_answer(predicted_answer, task['answer'])
        print(f"Correctness: {correctness}")
        
        # Update result with computed fields
        result.update({
            "predicted_answer": predicted_answer,
            "time_elapsed": round(llm_elapsed, 1),
            "correctness": correctness
        })
        
    except Exception as e:
        # Log other errors with details
        print(f"Unexpected error in single_task for {llm.name} and task id {task['id']}: {type(e).__name__}: {str(e)}")
        llm_elapsed = time.time() - llm_start
        print(f"Time elapsed since function start: {llm_elapsed:.1f}s")
        # Return result to continue processing other tasks, but mark this one as incorrect
        result.update({
            "predicted_answer": "ERROR",
            "time_elapsed": round(llm_elapsed, 1),
            "correctness": 0.0
        })
    
    return result

%choose multi_task
@kbench.task(name=TASK_NAME)
def multi_task(llm, df) -> float:
    print(f"Starting evaluation with {len(df)} tasks")
    print(f"LLM: {llm.name}")

    # Convert dataframe rows to list of dicts for evaluation
    tasks_list = df.to_dict('records')
    
    with kbench.client.enable_cache():
        runs = single_task.evaluate(
            stop_condition=lambda runs: len(runs) == len(tasks_list),
            max_attempts=1,
            retry_delay=15,
            llm=[llm],
            task=tasks_list,  # Pass tasks as list of dicts
            n_jobs=1,  # Use single job 
            timeout=None,  # No timeout - let it complete
            remove_run_files=True,  # Optionally remove sub runs files.
        )
    
    print(f"Evaluation complete, processing results...")
    eval_df = runs.as_dataframe()
    
    if eval_df.empty:
        print("WARNING: eval_df is empty!")
        return 0.0

    # Extract just the results (without framework metadata)
    results_list = eval_df.result.tolist()
    
    # Print results in JSON format
    print("\nEvaluation Results (JSON format):")
    print(json.dumps(results_list, indent=2))
    print()

    correctness_series = eval_df.result.str.get("correctness")

    accuracy = float(correctness_series.mean())  # Use float() to convert from np.float
    print(f"Final result: {accuracy:.2%}")  
    
    return accuracy

## Load tasks from dataset

In [ ]:
with open(f"/kaggle/input/datasets/joosthazelzet/spatial-reasoning-tasks/orthographic_dataset_{DATASET_VARIANT}_tasks.json") as f:
    dataset = json.load(f)

df_tasks = pd.DataFrame(dataset['tasks'][:TASKS_SIZE])
print(f"{len(df_tasks)} tasks loaded from dataset")

## Test single sample

In [ ]:
sample = df_tasks.iloc[0]
sample

### Preview views and prompt

In [ ]:
md_views = format_input_views(sample["input"])
#print(md_views)

In [ ]:
md_prompt = build_prompt(input_views=sample["input"])
#print(md_prompt)

In [ ]:
#single_task.run(USE_MODEL, sample)

## Evaluate the full dataset

In [ ]:
multi_task.run(USE_MODEL, df_tasks)